In [2]:

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field


In [3]:
load_dotenv()

True

In [4]:
model = ChatOpenAI(model="gpt-5.4-mini")

In [5]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review")


In [6]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')


In [7]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [8]:
prompt = "What is the sentiment of the following review - The software doesnt work as advertised and the screen apperance is not fluid"

In [9]:
structured_model2.invoke(prompt)

DiagnosisSchema(issue_type='Bug', tone='frustrated', urgency='medium')

In [10]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["postive", "negative"]
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state: ReviewState):
    prompt = f'fOR THE following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment